# DDoS Auswertung

## Imports

In [1]:
import subprocess
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

## Globale Parameter

In [2]:
SERVER_IP = "141.22.28.227"

## 1. Parsen der Dateien

In [3]:
schema_overrides = {
        "frame.time": pl.Datetime,
        "frame.time_epoch": pl.Float64,
        "frame.len": pl.Int64,
        "tcp.srcport": pl.Int64,
        "tcp.dstport": pl.Int64,
        "udp.srcport": pl.Int64,
        "udp.dstport": pl.Int64,
}

SMALL_FIELDS = [
    "frame.time",
    "frame.time_epoch",
    "frame.len",
    "frame.protocols",
    "ip.proto",
    "ip.src",
    "ip.dst",
    "tcp.srcport",
    "tcp.dstport",
    "udp.srcport",
    "udp.dstport",
]

In [4]:
def pcap_to_parquet(input_file: Path, output_file: Path, batch_size: int = 500_000) -> None:
    csv_path = output_file.with_suffix(".csv")
    cmd = ["tshark", "-r", str(input_file), "-T", "fields"]
    for f in SMALL_FIELDS:
        cmd += ["-e", f]
    cmd += ["-E", "header=y", "-E", "separator=,", "-E", "quote=d", "-E", "occurrence=f"]

    with open(csv_path, "wb") as f:
        result = subprocess.run(cmd, stdout=f, stderr=subprocess.PIPE)
    if result.returncode != 0:
        csv_path.unlink(missing_ok=True)
        raise RuntimeError(result.stderr.decode())

    lf = pl.scan_csv(csv_path, schema_overrides=schema_overrides, null_values=[""])

    writer = None
    try:
        for batch in lf.collect_batches(chunk_size=batch_size):
            table = batch.to_arrow()
            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema)
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    csv_path.unlink()

In [6]:
input_file = Path("data/raw/2025-08-13_15-38CEST_ddos-event_full-packets_100MiB.pcap.zst")
output_file = Path("data/interim/2025-08-13_15-38CEST_ddos-event_full-packets_100MiB.parquet")
df = pcap_to_parquet(input_file, output_file)

ComputeError: could not parse `"Aug 13, 2025 13:28:29.087374000 UTC"` as dtype `datetime[μs]` at column 'frame.time' (column number 1)

The current offset in the file is 0 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `"Aug 13, 2025 13:28:29.087374000 UTC"` to the `null_values` list.

Original error: ```could not find a 'date/datetime' pattern for 'Aug 13, 2025 13:28:29.087374000 UTC'```

## 2. Analyse der Daten

### 2.1 Überblick über die Daten (raw)

In [ ]:
lf = pl.scan_parquet(output_file)
df_sample = lf.limit(100).collect().sample(10)
df_sample

### 2.1.2 hinzufügen von Daten

In [ ]:
# read IANA list to convert number to text
lf_pro_nb = (
    pl.scan_csv("data/external/protocol-numbers-1.csv", ignore_errors=True)
      .select("Decimal", "Keyword")
).rename({"Keyword": "ip.proto.name", "Decimal": "ip.proto.num"})
lf_pro_nb.head(10).collect()

In [ ]:
lf = (
    lf.join(lf_pro_nb, left_on="ip.proto", right_on="ip.proto.num", how="left")
    .rename({"ip.proto": "ip.proto.num"})
    .pipe(lambda lf: lf.select(
    (cols := lf.collect_schema().names(), cols.insert(5, cols.pop(cols.index("ip.proto.name"))), cols)[-1]
))
)

lf.head(10).collect()

### 2.2 Datenübersicht

#### 2.2.1 Incoming vs. Outgoing

In [ ]:
def split_direction(lf: pl.LazyFrame, server_ip):
    lf_out = lf.filter(pl.col("ip.src") == server_ip)
    lf_in = lf.filter(pl.col("ip.dst") == server_ip)
    lf_null = lf.filter(
        (pl.col("ip.dst") != server_ip) & (pl.col("ip.src") != server_ip)
    )
    lf_null = lf.filter(
        pl.col("ip.src").is_null() | pl.col("ip.dst").is_null()
    )
    print("lf_out:")
    lf_out.head(10).collect().show()
    
    return lf_out, lf_in, lf_null

In [ ]:
lf_out, lf_in, lf_null = split_direction(lf, SERVER_IP)

#### 2.2.2 Transport Protokoll

In [ ]:
def split_transport(lf: pl.LazyFrame) -> tuple[pl.LazyFrame, ...]:
    protos = lf.select("ip.proto.num").unique().collect()["ip.proto.num"].to_list()
    return tuple(
        lf.filter(
            pl.col("ip.proto.num").is_null() if proto is None
            else pl.col("ip.proto.num") == proto
        )
        for proto in protos
    )

In [ ]:
counts = (
    lf.group_by("ip.proto.num")
    .agg(pl.len().alias("count"))
    .collect()
    .sort("count", descending=True)
)

proto_names = {6: "TCP", 17: "UDP", 1: "ICMP", 88: "EIGRP", None: "unknown"}
total = counts["count"].sum()

labels = [
    f"{proto_names.get(p, str(p))}: {c/total*100:.1f}%"
    for p, c in zip(counts["ip.proto.num"], counts["count"])
]

fig, ax = plt.subplots(figsize=(7, 6))
wedges, _ = ax.pie(counts["count"])
ax.set_title("Pakete nach Transportprotokoll")
ax.legend(wedges, labels, title="Protokoll", loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
total_packets: int  = lf.select(pl.len()).collect().item()
total_bytes: int = lf.select(pl.col("frame.len").sum()).collect().item()

t_start = lf.select(pl.col("frame.time").min()).collect().item()
t_end = lf.select(pl.col("frame.time").max()).collect().item()
duration_s = (t_end - t_start).total_seconds()

print(f"Pakete gesamt : {total_packets:>12,}")
print(f"Bytes gesamt  : {total_bytes:>12,}  ({total_bytes/1e6:.1f} MB)")
print(f"Zeitraum      : {t_start}  →  {t_end}")
print(f"Dauer         : {duration_s:.1f} s")
print(f"Ø Paketrate   : {total_packets/duration_s:>10,.1f} pkt/s")
print(f"Ø Bitrate     : {total_bytes*8/duration_s/1e6:>10,.1f} Mbit/s")

### Packets/Bytes per Second

In [ ]:
df_in_per_sec = (
    lf_in.with_columns(pl.col("frame.time_epoch").cast(pl.Int64).alias("ts"))
    .group_by(pl.col("ts"))
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("ts")
    .collect()
)

df_in_per_sec.head(20)

## 5. Top-Talker (meiste Pakete / meiste Bytes)

In [ ]:
top_talkers = (
    lf.group_by("ip.src")
    .agg([
        pl.len().alias("packets"),
        pl.col("frame.len").sum().alias("bytes"),
    ])
    .sort("bytes", descending=True)
    .collect()
)

top_talkers.head(20)

## 7. Amplification-Metriken (BAF/PAF), optional

Falls Request/Response anhand `coap.code` oder Port-Richtung unterschieden werden können, hier die Grundstruktur. An eure CoAP-Logik anpassen (z. B. Server-Port als Filter, GET-Request vs. Response).

In [ ]:
SERVER_PORT = 5683  # ggf. anpassen

requests = lf.filter(pl.col("udp.dstport") == SERVER_PORT)
responses = lf.filter(pl.col("udp.srcport") == SERVER_PORT)

req_stats = requests.select([
    pl.len().alias("req_packets"),
    pl.col("frame.len").sum().alias("req_bytes"),
]).collect()

resp_stats = responses.select([
    pl.len().alias("resp_packets"),
    pl.col("frame.len").sum().alias("resp_bytes"),
]).collect()

baf = resp_stats["resp_bytes"][0] / req_stats["req_bytes"][0]
paf = resp_stats["resp_packets"][0] / req_stats["req_packets"][0]

print(f"BAF (Bandwidth Amplification Factor): {baf:.2f}")
print(f"PAF (Packet Amplification Factor):   {paf:.2f}")